# OxtaMem â€” prova de qualidade end-to-end (T4)

**O modelo responde melhor quando o OxtaMem lhe entrega o fato certo**, numa tarefa
onde os fatos excedem a janela de contexto? Mesmo modelo (treinado em recall
associativo MQAR). No teste: **modelo-so** (janela recente de C fatos) vs
**modelo+OxtaMem** (recupera o fato via `forward_trunk` e injeta).

Robusto: erros de build aparecem inteiros. A validacao oficial exige OxtaMem
nativo; o fallback numpy fica disponivel apenas para diagnostico. Runtime: **T4**. Rode em ordem.

## 1 â€” Helpers + GPU + clone (branch `nsos-gpu-phases12`)

In [ ]:
import os, sys, subprocess, pathlib, glob, shutil, base64
def sh(cmd, cwd=None, extra_env=None, quiet=False):
    """Roda um comando mostrando o ERRO REAL (stdout+stderr) se falhar."""
    env = dict(os.environ);
    if extra_env: env.update(extra_env)
    # string -> roda no shell (necessario p/ pipes como 'curl | sh'); lista -> direto
    r = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, shell=isinstance(cmd, str))
    if r.returncode != 0:
        shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
        print("COMANDO FALHOU:", shown)
        if r.stdout.strip(): print("--- stdout (fim) ---\n" + r.stdout[-3500:])
        if r.stderr.strip(): print("--- stderr (fim) ---\n" + r.stderr[-3500:])
        raise RuntimeError(f"exit {r.returncode}: {shown}")
    elif not quiet and r.stdout.strip():
        print(r.stdout[-800:])
    return r

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
REPO = '/content/reimagined'; BRANCH = 'nsos-gpu-phases12'
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/vitorGgC569/reimagined.git')
TOKEN = None
try:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            TOKEN = userdata.get(key)
            if TOKEN: print('token', key, 'ok'); break
        except Exception: pass
except Exception as e:
    print('userdata indisponivel:', e)
if not TOKEN and 'github.com' in REPO_URL:
    import getpass
    TOKEN = getpass.getpass('GitHub token (entrada oculta): ')
assert TOKEN or 'github.com' not in REPO_URL, 'repo privado: configure o Colab Secret GITHUB_TOKEN (icone chave)'
git_env = {}
if TOKEN:
    auth = base64.b64encode(f'oauth2:{TOKEN}'.encode()).decode()
    git_env = {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader',
               'GIT_CONFIG_VALUE_0': f'Authorization: Basic {auth}'}
if not pathlib.Path(REPO, '.git').exists():
    if pathlib.Path(REPO).exists(): shutil.rmtree(REPO)
    sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO], extra_env=git_env)
else:
    sh(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', BRANCH], extra_env=git_env)
    sh(['git', '-C', REPO, 'reset', '--hard', 'FETCH_HEAD'])
commit = sh(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], quiet=True).stdout.strip()
os.environ['REPO'] = REPO
print('repo pronto:', REPO, '| branch', BRANCH, '| commit', commit)

## 2 â€” Build do `nsos_ext` (CUDA â€” receita do bootstrap oficial)

In [ ]:
BUILD = f'{REPO}/OXN/nsos/build-t4'
sh(['apt-get', 'update', '-qq'])
sh(['apt-get', 'install', '-y', '-qq', 'cmake', 'ninja-build', 'build-essential'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'pybind11>=2.10', 'numpy'])
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout
arch = next((v for k, v in {'T4':'75','V100':'70','L4':'89','A100':'80','H100':'90'}.items() if k in gpu), '75')
print('GPU', gpu.strip(), '-> sm_' + arch)
sh(['cmake', '-G', 'Ninja', '-S', f'{REPO}/OXN/nsos', '-B', BUILD,
    '-DCMAKE_BUILD_TYPE=Release', '-DNSOS_ENABLE_CUDA=ON', '-DNSOS_BUILD_PYTHON=ON',
    '-DNSOS_BUILD_TESTS=OFF', '-DNSOS_BUILD_CLI=OFF', '-DNSOS_BUILD_API=OFF',
    '-DNSOS_BUILD_OXTAMEM=OFF', f'-DNSOS_CUDA_ARCHITECTURES={arch}'])
sh(['cmake', '--build', BUILD, '--parallel', str(os.cpu_count() or 2), '--target', 'nsos_ext'])
so = glob.glob(f'{BUILD}/**/nsos_ext*.so', recursive=True)
assert so, 'nsos_ext nao compilou'
sys.path.insert(0, str(pathlib.Path(so[0]).parent))
import nsos_ext as nsos
print('nsos_ext OK:', so[0])

## 3 â€” Memoria vetorial: OxtaMem nativo (rust+maturin)

A validacao oficial exige o OxtaMem nativo. O store numpy com retrieval cosine
identico permanece apenas como ferramenta diagnostica (`REQUIRE_NATIVE = False`).

In [ ]:
import numpy as np
class NumpyMemStore:
    """Fallback de memoria vetorial: cosine KNN exato. Mesma interface do
    PyGeodesicEngine usada aqui (write_with_vector / search_similar / flush)."""
    def __init__(self): self.vecs=[]; self.values=[]
    def set_sync_on_write(self, b): pass
    def flush(self): pass
    def write_with_vector(self, token_id, value, vector):
        v = np.asarray(vector, dtype=np.float32); v = v/(np.linalg.norm(v)+1e-8)
        self.vecs.append(v); self.values.append(bytes(value)); return len(self.values)-1
    def search_similar(self, vector, k):
        if not self.vecs: return []
        q = np.asarray(vector, dtype=np.float32); q = q/(np.linalg.norm(q)+1e-8)
        sims = np.stack(self.vecs) @ q
        idx = np.argsort(-sims)[:k]
        return [self.values[i] for i in idx]

REQUIRE_NATIVE = True
USE_OXTA = False; PyGeodesicEngine = None
try:
    sh(['apt-get', 'install', '-y', '-qq', 'patchelf'])   # maturin repair dep
    if shutil.which('cargo') is None:
        print('instalando rust toolchain...')
        sh('curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal')
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'])
    # --compatibility linux evita o repair manylinux (que exige patchelf/docker e quebra no Colab)
    sh(['maturin', 'build', '--release', '--compatibility', 'linux', '--interpreter', sys.executable],
       cwd=f'{REPO}/modules/oxtamem')
    whl = glob.glob(f'{REPO}/modules/oxtamem/oxta_engine/target/wheels/oxta_mem-*.whl')
    assert whl, 'wheel nao gerado'
    sh([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', whl[0]])
    from oxta_mem.native import PyGeodesicEngine
    USE_OXTA = True
    print('>>> OxtaMem NATIVO OK:', whl[0])
except Exception as e:
    print('>>> OxtaMem nativo indisponivel. Motivo:', repr(e)[:400])
    if REQUIRE_NATIVE:
        raise RuntimeError('Validacao oficial exige OxtaMem nativo; fallback numpy nao e prova do OxtaMem') from e
print('memoria em uso:', 'OxtaMem nativo' if USE_OXTA else 'NumpyMemStore (cosine KNN)')

In [ ]:
sh(['cargo', 'test', '--release', '--locked'], cwd=f'{REPO}/modules/oxtamem/oxta_engine')

import gc, tempfile
root_gate = tempfile.mkdtemp(prefix='oxta_gate_', dir='/content')
db_gate = os.path.join(root_gate, 'store.db')
gate = PyGeodesicEngine(db_gate, 8)
gate.set_sync_on_write(False)
vectors_gate = []
for i in range(10):
    v = np.zeros(128, dtype=np.float32); v[i] = 1.0
    vectors_gate.append(v.tolist())
    gate.write_with_vector(str(i), bytes([i]), v.tolist())
gate.flush()
assert all(gate.read_latest(str(i)) == bytes([i]) for i in range(10))
assert all(gate.search_similar(vectors_gate[i], 1)[0] == bytes([i]) for i in range(10))
del gate; gc.collect()

gate = PyGeodesicEngine(db_gate, 8)
assert all(gate.read_latest(str(i)) == bytes([i]) for i in range(10))
assert all(gate.search_similar(vectors_gate[i], 1)[0] == bytes([i]) for i in range(10))
del gate; gc.collect(); shutil.rmtree(root_gate, ignore_errors=True)
print('>>> OxtaMem alignment/persistence gate: 10/10 OK')

## 4 â€” Tarefa de memoria (MQAR) + treino

Modelo aprende lookup associativo em contexto (`k1 v1 ... [Q] kq` -> `vq`), com
comprimento variavel 1..8 -> robusto tanto ao 1 fato injetado quanto a janela cheia.

In [ ]:
import numpy as np, random, math, time
GPU = nsos.Device.GPU
os.environ['NSOS_ATTN_LAST_GUARD'] = '0'   # hibrido [M,A,M,A] (atencao ajuda recall)
QUERY = 1; V0, V1 = 2, 66; K0, K1 = 66, 1266; VOCAB = 1266   # values byte-safe (<256); muitas keys
D_MODEL, D_STATE = 128, 64; N_PAIRS = 8; BATCH = 32; LR = 2e-3; STEPS = 2500; SEED = 0

def _set(c, k, v):
    try: setattr(c, k, v)
    except Exception: pass
def build():
    c = nsos.ModelConfig(); c.num_layers=4; c.d_model=D_MODEL; c.vocab_size=VOCAB
    c.n_heads=4; c.n_kv_heads=2; c.max_context_tokens=1024
    c.use_moe=False; c.use_kan=False; c.use_ttt=False; c.dropout=0.0
    _set(c,'mamba2_faithful',True); _set(c,'mamba_expand',2); _set(c,'mamba_head_dim',64)
    _set(c,'mamba_state_expansion',D_STATE); _set(c,'mamba_n_groups',1)
    c.attention_period=2; c.attention_slot=1
    m = nsos.JambaModel(c, GPU); m.to(GPU); return m

def mqar_seq(rng, n_pairs):
    keys = rng.sample(range(K0, K1), n_pairs)
    vals = [rng.randrange(V0, V1) for _ in range(n_pairs)]
    seq = []
    for k, v in zip(keys, vals): seq += [k, v]
    qi = rng.randrange(n_pairs); seq += [QUERY, keys[qi]]
    return seq, vals[qi]

def batch(rng, n_pairs):
    ps, an = [], []
    for _ in range(BATCH):
        s, v = mqar_seq(rng, n_pairs); ps.append(s); an.append([v])
    return ps, an

nsos.set_seed(SEED); model = build()
tr = nsos.Trainer(model, LR); tr.warmup_steps=100; tr.total_training_steps=STEPS
tr.phase_scheduler.progressive_qat_enabled=False
rng = random.Random(7); model.set_training_mode(True); t0=time.time(); run=0.0
for s in range(STEPS):
    npair = rng.randint(1, N_PAIRS)
    ps, an = batch(rng, npair); run += tr.train_supervised_batch(ps, an)
    if (s+1) % 500 == 0: print(f'  step {s+1}/{STEPS} loss~{run/500:.3f}'); run=0.0
model.set_training_mode(False)
nsos_train_s = time.time() - t0
print(f'treino ok em {nsos_train_s:.1f}s')

## 5 â€” Sanity: o modelo aprendeu o lookup em contexto?

In [ ]:
def _argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())
vr = random.Random(999); ok = 0; T = 300
for _ in range(T):
    s, v = mqar_seq(vr, N_PAIRS); ok += (_argmax_last(s) == v)
acc = ok / T
print(f'MQAR in-context acc (n_pairs={N_PAIRS}) = {acc:.3f}')
assert acc > 0.6, 'modelo nao aprendeu o lookup â€” aumente STEPS ou cheque o build'

## 6 â€” Prova end-to-end: modelo-so vs modelo+OxtaMem (fatos >> contexto)

In [ ]:
def argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())

def embed_key(k):
    tr_ = np.asarray(model.forward_trunk([QUERY, k]).cpu().numpy()).reshape(-1, D_MODEL)
    return tr_[-1].astype(np.float32).tolist()

def make_store():
    if USE_OXTA:
        import tempfile
        root = tempfile.mkdtemp(prefix='oxta_qa_', dir='/content')
        db = os.path.join(root, 'qa_store.db')
        eng = PyGeodesicEngine(db, 8); eng.set_sync_on_write(False); return eng, root
    return NumpyMemStore(), None

def run_eval(N, C=8, n_query=200, seed=0):
    rng = random.Random(seed)
    keys = rng.sample(range(K0, K1), min(N, K1 - K0)); N = len(keys)
    vals = {k: rng.randrange(V0, V1) for k in keys}
    store, store_root = make_store()
    for k in keys:
        store.write_with_vector(f'k{k}', bytes([vals[k]]), embed_key(k))
    store.flush()
    order = keys[:]                       # ordem de chegada; ultimos C = janela do modelo-so
    qrng = random.Random(seed * 7 + 1)
    qs = [qrng.randrange(N) for _ in range(n_query)]
    alone = oxta = retr = 0
    for qi in qs:
        qk = order[qi]; true_v = vals[qk]
        window = order[max(0, N - C):]
        seq = []
        for k in window: seq += [k, vals[k]]
        seq += [QUERY, qk]
        alone += (argmax_last(seq) == true_v)      # modelo-so (so a janela)
        got = store.search_similar(embed_key(qk), 1)
        rv = got[0][0] if got else -1              # value recuperado da memoria
        retr += (rv == true_v)
        oxta += (argmax_last([qk, rv, QUERY, qk]) == true_v)   # modelo + fato injetado
    result = dict(N=N, C=C, alone=alone/len(qs), oxta=oxta/len(qs), retr=retr/len(qs))
    if store_root:
        import gc
        del store; gc.collect(); shutil.rmtree(store_root, ignore_errors=True)
    return result

print(f'store: {"OxtaMem nativo" if USE_OXTA else "NumpyMemStore"}')
print(f'{"N":>6} {"C(janela)":>10} {"modelo-so":>11} {"modelo+OxtaMem":>15} {"(retr@1)":>10}')
results = []
for N in (50, 200, 1000):
    r = run_eval(N, C=8)
    results.append(r)
    print(f'{r["N"]:>6} {r["C"]:>10} {r["alone"]:>11.3f} {r["oxta"]:>15.3f} {r["retr"]:>10.3f}')
assert USE_OXTA, 'validacao oficial exige OxtaMem nativo'
assert min(r['retr'] for r in results) >= 0.99, 'recall exato do OxtaMem abaixo de 99%'
assert results[-1]['oxta'] > results[-1]['alone'] + 0.30, 'OxtaMem nao melhorou o caso N=1000'

## 7 â€” Leitura

- **modelo-so**: so acerta se o fato caiu na janela dos ultimos C -> acc ~ C/N, cai com N.
- **modelo+OxtaMem**: recupera o fato certo da memoria O(N) e injeta -> acc ~ retrieval (~1.0).
- `modelo+OxtaMem >> modelo-so` em N grande = **o OxtaMem faz o modelo responder melhor**
  (qualidade end-to-end, nao so capacidade). O store usado (nativo/numpy) faz o MESMO
  retrieval cosine â€” a conclusao independe da implementacao.

## 8 — Jamba oficial vs NSOS, com e sem OxtaMem

Comparacao sintetica controlada usando a implementacao oficial JambaForCausalLM da Hugging Face/AI21.

Para isolar o nucleo Mamba+Attention, ambos usam quatro camadas [Mamba, Attention, Mamba, Attention], d_model=128, d_state=64, batch 32, 2.500 passos, mesmo gerador MQAR, mesmo otimizador e mesmo scheduler. MoE fica desativado nos dois bracos.

Fontes oficiais:
- https://huggingface.co/docs/transformers/model_doc/jamba
- https://arxiv.org/abs/2403.19887


In [ ]:
import torch, transformers
try:
    from transformers import JambaConfig, JambaForCausalLM
except ImportError:
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.48'])
    from transformers import JambaConfig, JambaForCausalLM

print('torch', torch.__version__, '| transformers', transformers.__version__)
hf_cfg = JambaConfig(
    vocab_size=VOCAB, hidden_size=D_MODEL, intermediate_size=4*D_MODEL,
    num_hidden_layers=4, num_attention_heads=4, num_key_value_heads=2,
    attn_layer_period=2, attn_layer_offset=1,
    mamba_d_state=D_STATE, mamba_d_conv=4, mamba_expand=2,
    num_experts=1, num_experts_per_tok=1,
    hidden_act='silu', attention_dropout=0.0,
    rms_norm_eps=1e-5, use_cache=False, use_mamba_kernels=False,
    tie_word_embeddings=True, max_position_embeddings=1024
)
torch.manual_seed(SEED)
hf_probe = JambaForCausalLM(hf_cfg).cuda().float().eval()
with torch.no_grad():
    z = hf_probe(torch.tensor([[66, 2, QUERY, 66]], device='cuda'), use_cache=False).logits
print('HF Jamba probe', tuple(z.shape), '| params', f'{sum(p.numel() for p in hf_probe.parameters()):,}')
print('camadas:', [type(x).__name__ for x in hf_probe.model.layers])
del z, hf_probe
torch.cuda.empty_cache()


In [ ]:
import gc, time, math
import torch.nn.functional as F

# Comparacao controlada: mesma topologia, dados, batch, passos, LR, AdamW e scheduler.
torch.manual_seed(SEED)
hf_model = JambaForCausalLM(hf_cfg).cuda().float()
hf_params = sum(p.numel() for p in hf_model.parameters())
nsos_params = sum(int(p.data.size) for p in model.parameters())
print(f'PARAMS | HF Jamba oficial={hf_params:,} | NSOS={nsos_params:,}')

opt = torch.optim.AdamW(hf_model.parameters(), lr=LR, betas=(0.9, 0.999),
                        eps=1e-8, weight_decay=0.01)
def lr_at(step):
    if step <= 100:
        return LR * step / 100.0
    progress = min(max((step - 100) / (STEPS - 100), 0.0), 1.0)
    return LR * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

rng_hf = random.Random(7)
hf_model.train()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
t_hf = time.time()
run_loss = 0.0
for step in range(STEPS):
    n_pair = rng_hf.randint(1, N_PAIRS)
    seqs, ys = [], []
    for _ in range(BATCH):
        seq, value = mqar_seq(rng_hf, n_pair)
        seqs.append(seq)
        ys.append(value)
    ids = torch.tensor(seqs, device='cuda', dtype=torch.long)
    target = torch.tensor(ys, device='cuda', dtype=torch.long)
    for group in opt.param_groups:
        group['lr'] = lr_at(step)
    logits = hf_model(ids, use_cache=False).logits[:, -1, :]
    loss = F.cross_entropy(logits, target)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(hf_model.parameters(), 1.0)
    opt.step()
    run_loss += float(loss.detach())
    if (step + 1) % 500 == 0:
        print(f' HF step {step+1}/{STEPS} loss~{run_loss/500:.3f} lr={lr_at(step):.6f}')
        run_loss = 0.0

hf_train_s = time.time() - t_hf
hf_peak_mb = torch.cuda.max_memory_allocated() / 1e6
hf_model.eval()
print(f'HF TREINO OK | {hf_train_s:.1f}s | pico torch={hf_peak_mb:.1f} MB')

def fixed_eval(n_pair, seed, total=300):
    rr = random.Random(seed)
    pairs = [mqar_seq(rr, n_pair) for _ in range(total)]
    return [x[0] for x in pairs], [x[1] for x in pairs]

@torch.no_grad()
def hf_predict_many(seqs, chunk=128):
    out = []
    for start in range(0, len(seqs), chunk):
        ids = torch.tensor(seqs[start:start+chunk], device='cuda', dtype=torch.long)
        out.extend(hf_model(ids, use_cache=False).logits[:, -1, :].argmax(-1).cpu().tolist())
    return out

def nsos_predict_many(seqs):
    return [argmax_last(seq) for seq in seqs]

mqar_compare = []
print()
print('MQAR EM CONTEXTO (mesmos 300 exemplos)')
print(f'{"pares":>7} {"HF oficial":>12} {"NSOS":>12}')
for n_pair in (1, 4, 8):
    seqs, ys = fixed_eval(n_pair, 9000+n_pair)
    ph = hf_predict_many(seqs)
    pn = nsos_predict_many(seqs)
    ah = sum(a == b for a,b in zip(ph,ys)) / len(ys)
    an = sum(a == b for a,b in zip(pn,ys)) / len(ys)
    mqar_compare.append(dict(n_pairs=n_pair, hf=ah, nsos=an))
    print(f'{n_pair:>7} {ah:>12.3f} {an:>12.3f}')

def memory_sequences(N, C=8, n_query=200, seed=0):
    rr = random.Random(seed)
    keys = rr.sample(range(K0, K1), min(N, K1-K0))
    N = len(keys)
    vals = {k: rr.randrange(V0, V1) for k in keys}
    qr = random.Random(seed*7+1)
    qs = [qr.randrange(N) for _ in range(n_query)]
    window = keys[max(0, N-C):]
    seqs, ys = [], []
    for qi in qs:
        qk = keys[qi]
        seq = []
        for k in window:
            seq += [k, vals[k]]
        seq += [QUERY, qk]
        seqs.append(seq)
        ys.append(vals[qk])
    return seqs, ys

memory_compare = []
print()
print('MEMORIA LONGA, janela C=8 (mesmas 200 consultas)')
print(f'{"N":>6} {"HF sem mem":>12} {"NSOS sem mem":>14} {"NSOS+Oxta":>13} {"retr@1":>9}')
for N in (50, 200, 1000):
    seqs, ys = memory_sequences(N, C=8, n_query=200, seed=0)
    ph = hf_predict_many(seqs)
    hf_acc = sum(a == b for a,b in zip(ph,ys)) / len(ys)
    ro = run_eval(N, C=8, n_query=200, seed=0)
    row = dict(N=N, hf=hf_acc, nsos=ro['alone'], oxta=ro['oxta'], retr=ro['retr'])
    memory_compare.append(row)
    print(f'{N:>6} {hf_acc:>12.3f} {ro["alone"]:>14.3f} {ro["oxta"]:>13.3f} {ro["retr"]:>9.3f}')

benchmark_summary = dict(
    reference='Hugging Face JambaForCausalLM (implementacao oficial AI21/HF)',
    schedule='[Mamba, Attention, Mamba, Attention]',
    moe='desligado em ambos para isolar Mamba+Attention',
    d_model=D_MODEL, layers=4, d_state=D_STATE, batch=BATCH, steps=STEPS, lr=LR,
    params_hf=hf_params, params_nsos=nsos_params,
    train_seconds_hf=hf_train_s, train_seconds_nsos=nsos_train_s,
    torch_peak_mb=hf_peak_mb,
    mqar=mqar_compare, memory=memory_compare,
)
assert memory_compare[-1]['oxta'] >= 0.99
assert memory_compare[-1]['oxta'] > max(memory_compare[-1]['hf'], memory_compare[-1]['nsos']) + 0.30
print()
print('VALIDACAO FINAL: APROVADA')
print(benchmark_summary)


In [ ]:
EVAL_SEEDS = [999, 9008, 12345, 31415, 27182]
robust_mqar = []
print('MQAR MULTISSEMENTE: media +- desvio (5 x 300 exemplos)')
print(f'{"pares":>7} {"HF oficial":>20} {"NSOS":>20}')
for n_pair in (1, 4, 8):
    ahs, ans = [], []
    for es in EVAL_SEEDS:
        seqs, ys = fixed_eval(n_pair, es, total=300)
        ph = hf_predict_many(seqs)
        pn = nsos_predict_many(seqs)
        ahs.append(sum(a == b for a,b in zip(ph,ys)) / len(ys))
        ans.append(sum(a == b for a,b in zip(pn,ys)) / len(ys))
    row = dict(n_pairs=n_pair, hf_mean=float(np.mean(ahs)), hf_std=float(np.std(ahs)),
               nsos_mean=float(np.mean(ans)), nsos_std=float(np.std(ans)),
               hf_runs=ahs, nsos_runs=ans)
    robust_mqar.append(row)
    print(f'{n_pair:>7} {row["hf_mean"]:>8.3f} +- {row["hf_std"]:<7.3f} {row["nsos_mean"]:>8.3f} +- {row["nsos_std"]:<7.3f}')

robust_memory = []
print()
print('MEMORIA LONGA MULTISSEMENTE: media +- desvio (3 x 200 consultas)')
print(f'{"N":>6} {"HF sem mem":>20} {"NSOS sem mem":>20} {"NSOS+Oxta":>20} {"retr@1":>12}')
for N in (50, 200, 1000):
    hfs, nss, oxs, rts = [], [], [], []
    for es in (0, 1, 2):
        seqs, ys = memory_sequences(N, C=8, n_query=200, seed=es)
        ph = hf_predict_many(seqs)
        hfs.append(sum(a == b for a,b in zip(ph,ys)) / len(ys))
        ro = run_eval(N, C=8, n_query=200, seed=es)
        nss.append(ro['alone'])
        oxs.append(ro['oxta'])
        rts.append(ro['retr'])
    row = dict(N=N, hf_mean=float(np.mean(hfs)), hf_std=float(np.std(hfs)),
               nsos_mean=float(np.mean(nss)), nsos_std=float(np.std(nss)),
               oxta_mean=float(np.mean(oxs)), oxta_std=float(np.std(oxs)),
               retr_mean=float(np.mean(rts)), retr_std=float(np.std(rts)))
    robust_memory.append(row)
    print(f'{N:>6} {row["hf_mean"]:>8.3f} +- {row["hf_std"]:<7.3f} {row["nsos_mean"]:>8.3f} +- {row["nsos_std"]:<7.3f} {row["oxta_mean"]:>8.3f} +- {row["oxta_std"]:<7.3f} {row["retr_mean"]:>12.3f}')

benchmark_summary['robust_mqar'] = robust_mqar
benchmark_summary['robust_memory'] = robust_memory
assert min(x['oxta_mean'] for x in robust_memory) >= 0.99
assert robust_mqar[-1]['nsos_mean'] > robust_mqar[-1]['hf_mean'] + 0.30
print()
print('ROBUSTEZ MULTISSEMENTE: APROVADA')


## 9 — Resultado validado em Tesla T4

MQAR multissemente (media de 5 sementes, 300 exemplos por semente):

- 1 par: oficial 1,000; NSOS 1,000.
- 4 pares: oficial 0,259 +/- 0,025; NSOS 0,994 +/- 0,002.
- 8 pares: oficial 0,128 +/- 0,023; NSOS 0,955 +/- 0,014.

Memoria longa (media de 3 sementes, 200 consultas por semente):

- N=50: oficial sem memoria 0,020; NSOS sem memoria 0,132; NSOS+OxtaMem 1,000.
- N=200: oficial sem memoria 0,022; NSOS sem memoria 0,043; NSOS+OxtaMem 1,000.
- N=1000: oficial sem memoria 0,010; NSOS sem memoria 0,023; NSOS+OxtaMem 1,000.
- OxtaMem retrieval@1: 1,000 em todos os casos.

Recursos observados: Jamba oficial 1.354.896 parametros; NSOS 768.152 (43,3% menos). Treino no runtime validado: 117,4 s vs 34 s. O Jamba oficial rodou no fallback PyTorch sem kernels Mamba fundidos, portanto o tempo nao representa velocidade de producao.

Conclusao limitada ao teste: com o mesmo pequeno orcamento, o NSOS aprende recall sintetico mais rapido; o OxtaMem elimina a queda quando os fatos excedem a janela. Nao e uma alegacao de superioridade sobre checkpoints Jamba de producao, que usam escala, MoE e outra proporcao de atencao.


In [ ]:
# Jamba oficial com o MoE de producao: 16 experts, top-2, uma camada MoE a cada 2.
hf_moe_cfg = JambaConfig(
    vocab_size=VOCAB, hidden_size=D_MODEL, intermediate_size=4*D_MODEL,
    num_hidden_layers=4, num_attention_heads=4, num_key_value_heads=2,
    attn_layer_period=2, attn_layer_offset=1,
    mamba_d_state=D_STATE, mamba_d_conv=4, mamba_expand=2,
    num_experts=16, num_experts_per_tok=2,
    expert_layer_period=2, expert_layer_offset=1,
    router_aux_loss_coef=0.001, output_router_logits=True,
    hidden_act='silu', attention_dropout=0.0,
    rms_norm_eps=1e-5, use_cache=False, use_mamba_kernels=False,
    tie_word_embeddings=True, max_position_embeddings=1024
)
torch.manual_seed(SEED)
hf_moe_probe = JambaForCausalLM(hf_moe_cfg).cuda().float().eval()
with torch.no_grad():
    moe_out = hf_moe_probe(torch.tensor([[66, 2, QUERY, 66]], device='cuda'),
                           use_cache=False, output_router_logits=True)
print('HF Jamba MoE probe', tuple(moe_out.logits.shape),
      '| params', f'{sum(p.numel() for p in hf_moe_probe.parameters()):,}',
      '| aux_loss', getattr(moe_out, 'aux_loss', None))
print('camadas:', [type(x).__name__ + '/' + type(x.feed_forward).__name__
                   for x in hf_moe_probe.model.layers])
del moe_out, hf_moe_probe
torch.cuda.empty_cache()


In [ ]:
# Mesmo orcamento do teste anterior; inclui a perda auxiliar oficial do roteador.
torch.manual_seed(SEED)
hf_moe_model = JambaForCausalLM(hf_moe_cfg).cuda().float()
hf_moe_params = sum(p.numel() for p in hf_moe_model.parameters())
moe_opt = torch.optim.AdamW(hf_moe_model.parameters(), lr=LR, betas=(0.9, 0.999),
                            eps=1e-8, weight_decay=0.01)
moe_rng = random.Random(7)
hf_moe_model.train()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
moe_t0 = time.time()
moe_run_ce = 0.0
moe_run_aux = 0.0
for step in range(STEPS):
    n_pair = moe_rng.randint(1, N_PAIRS)
    seqs, ys = [], []
    for _ in range(BATCH):
        seq, value = mqar_seq(moe_rng, n_pair)
        seqs.append(seq)
        ys.append(value)
    ids = torch.tensor(seqs, device='cuda', dtype=torch.long)
    target = torch.tensor(ys, device='cuda', dtype=torch.long)
    for group in moe_opt.param_groups:
        group['lr'] = lr_at(step)
    out = hf_moe_model(ids, use_cache=False, output_router_logits=True)
    ce = F.cross_entropy(out.logits[:, -1, :], target)
    aux = out.aux_loss
    loss = ce + hf_moe_cfg.router_aux_loss_coef * aux
    moe_opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(hf_moe_model.parameters(), 1.0)
    moe_opt.step()
    moe_run_ce += float(ce.detach())
    moe_run_aux += float(aux.detach())
    if (step + 1) % 500 == 0:
        print(f' MoE step {step+1}/{STEPS} ce~{moe_run_ce/500:.3f} aux~{moe_run_aux/500:.3f} lr={lr_at(step):.6f}')
        moe_run_ce = 0.0
        moe_run_aux = 0.0

hf_moe_train_s = time.time() - moe_t0
hf_moe_peak_mb = torch.cuda.max_memory_allocated() / 1e6
hf_moe_model.eval()
print(f'HF Jamba MoE TREINO OK | {hf_moe_train_s:.1f}s | params={hf_moe_params:,} | pico torch={hf_moe_peak_mb:.1f} MB')

@torch.no_grad()
def hf_moe_predict_many(seqs, chunk=128):
    preds = []
    for start in range(0, len(seqs), chunk):
        ids = torch.tensor(seqs[start:start+chunk], device='cuda', dtype=torch.long)
        preds.extend(hf_moe_model(ids, use_cache=False).logits[:, -1, :].argmax(-1).cpu().tolist())
    return preds

moe_mqar = []
print()
print('MQAR MULTISSEMENTE: oficial denso vs oficial MoE vs NSOS')
print(f'{"pares":>7} {"oficial denso":>20} {"oficial MoE":>20} {"NSOS sem MoE":>20}')
for base_row, n_pair in zip(robust_mqar, (1, 4, 8)):
    scores = []
    for es in EVAL_SEEDS:
        seqs, ys = fixed_eval(n_pair, es, total=300)
        pred = hf_moe_predict_many(seqs)
        scores.append(sum(a == b for a,b in zip(pred, ys)) / len(ys))
    row = dict(n_pairs=n_pair, mean=float(np.mean(scores)), std=float(np.std(scores)), runs=scores)
    moe_mqar.append(row)
    print(f'{n_pair:>7} {base_row["hf_mean"]:>8.3f} +- {base_row["hf_std"]:<7.3f} {row["mean"]:>8.3f} +- {row["std"]:<7.3f} {base_row["nsos_mean"]:>8.3f} +- {base_row["nsos_std"]:<7.3f}')

moe_memory = []
print()
print('MEMORIA LONGA: oficial denso vs oficial MoE vs NSOS vs NSOS+OxtaMem')
print(f'{"N":>6} {"oficial denso":>15} {"oficial MoE":>15} {"NSOS":>12} {"NSOS+Oxta":>13}')
for base_row, N in zip(robust_memory, (50, 200, 1000)):
    scores = []
    for es in (0, 1, 2):
        seqs, ys = memory_sequences(N, C=8, n_query=200, seed=es)
        pred = hf_moe_predict_many(seqs)
        scores.append(sum(a == b for a,b in zip(pred, ys)) / len(ys))
    row = dict(N=N, mean=float(np.mean(scores)), std=float(np.std(scores)), runs=scores)
    moe_memory.append(row)
    print(f'{N:>6} {base_row["hf_mean"]:>15.3f} {row["mean"]:>15.3f} {base_row["nsos_mean"]:>12.3f} {base_row["oxta_mean"]:>13.3f}')

moe_summary = dict(
    experts=16, top_k=2, expert_period=2,
    params=hf_moe_params, train_seconds=hf_moe_train_s,
    torch_peak_mb=hf_moe_peak_mb,
    mqar=moe_mqar, memory=moe_memory,
)
assert hf_moe_params > hf_params
assert all(np.isfinite(x["mean"]) for x in moe_mqar + moe_memory)
print()
print('JAMBA OFICIAL COM MOE: EXECUCAO E AVALIACAO APROVADAS')
print(moe_summary)


## 10 — Jamba oficial com MoE completo

O braço oficial recebeu o MoE nativo do Jamba: 16 especialistas, top-2, uma camada MoE a cada duas camadas e perda auxiliar do roteador. O NSOS permaneceu sem MoE. Todo o restante ficou igual: mesmos dados, batch 32, 2.500 passos, scheduler e sementes.

Capacidade e tempo validados na Tesla T4:

- Jamba oficial denso: 1.354.896 parametros; 117,4 s.
- Jamba oficial com MoE: 7.257.232 parametros; 156,5 s.
- NSOS sem MoE: 768.152 parametros; 34 s.

MQAR, media de 5 sementes:

- 1 par: oficial denso 1,000; oficial MoE 0,998; NSOS 1,000.
- 4 pares: oficial denso 0,259; oficial MoE 0,241; NSOS 0,994.
- 8 pares: oficial denso 0,128; oficial MoE 0,127; NSOS 0,955.

Memoria longa, media de 3 sementes:

- N=50: oficial denso 0,020; oficial MoE 0,025; NSOS 0,132; NSOS+OxtaMem 1,000.
- N=200: oficial denso 0,022; oficial MoE 0,023; NSOS 0,043; NSOS+OxtaMem 1,000.
- N=1000: oficial denso 0,010; oficial MoE 0,013; NSOS 0,023; NSOS+OxtaMem 1,000.

Conclusao limitada ao teste: aumentar a capacidade com MoE nao resolveu o recall neste pequeno orcamento. O NSOS sem MoE continuou muito a frente; o OxtaMem continuou perfeito quando os fatos excederam a janela. Isto nao nega vantagens do MoE em grande escala: demonstra que capacidade adicional nao substituiu a dinamica de aprendizagem e a memoria externa neste teste controlado.
